# Demo 01 - NYC Taxi Trip Data

Notebook này dùng để chụp mẫu cho slide **Bộ dữ liệu sử dụng**. Demo chỉ đọc 2 file tháng mẫu để tránh scan toàn bộ 109M+ records khi thuyết trình.

## 4 phần nên chụp vào slide

1. Dataset overview
2. Schema / thuộc tính chính
3. Sample records
4. Quick summary theo taxi type

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebooks.utils.spark_session import get_spark
from pyspark.sql.functions import avg, col, lit, max as spark_max, min as spark_min, round as spark_round

spark = get_spark("MetroPulse Demo - Taxi Trip Data")
spark.sparkContext.setLogLevel("WARN")
spark.conf.get("spark.sql.session.timeZone")

## Load dữ liệu taxi mẫu

In [ ]:
yellow_path = PROJECT_ROOT / "data/raw/yellow_tripdata_2024-01.parquet"
green_path = PROJECT_ROOT / "data/raw/green_tripdata_2024-01.parquet"

yellow_raw = spark.read.parquet(str(yellow_path))
green_raw = spark.read.parquet(str(green_path))

# Chỉ lấy sample nhỏ để demo slide chạy nhanh, không scan cả file tháng.
yellow = yellow_raw.limit(5000).select(
    lit("yellow").alias("taxi_type"),
    col("tpep_pickup_datetime").alias("pickup_time"),
    col("tpep_dropoff_datetime").alias("dropoff_time"),
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "payment_type",
)

green = green_raw.limit(5000).select(
    lit("green").alias("taxi_type"),
    col("lpep_pickup_datetime").alias("pickup_time"),
    col("lpep_dropoff_datetime").alias("dropoff_time"),
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "payment_type",
)

taxi_demo = yellow.unionByName(green).cache()

# Trigger cache trên sample nhỏ để các cell sau phản hồi nhanh hơn.
taxi_demo.count()

## 1. Dataset overview

In [ ]:
overview = spark.createDataFrame(
    [
        ("Source", "NYC TLC / NYC Open Data"),
        ("Scope", "Yellow Taxi + Green Taxi"),
        ("Full dataset", "109,027,877 trips"),
        ("Raw size", "~6GB"),
        ("Demo files", "2024-01 yellow + 2024-01 green"),
    ],
    ["property", "value"],
)
overview.show(truncate=False)

## 2. Các thuộc tính chính

In [ ]:
attributes = spark.createDataFrame(
    [
        ("pickup_time", "timestamp", "Thời gian bắt đầu chuyến đi"),
        ("dropoff_time", "timestamp", "Thời gian kết thúc chuyến đi"),
        ("PULocationID", "integer", "Mã vùng đón khách"),
        ("DOLocationID", "integer", "Mã vùng trả khách"),
        ("passenger_count", "double", "Số hành khách"),
        ("trip_distance", "double", "Quãng đường chuyến đi"),
        ("fare_amount", "double", "Tiền cước"),
        ("tip_amount", "double", "Tiền tip"),
        ("payment_type", "integer", "Loại thanh toán"),
    ],
    ["column", "type", "meaning"],
)
attributes.show(truncate=False)

## 3. Sample records

In [ ]:
taxi_demo.select(
    "taxi_type",
    "pickup_time",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
).show(5, truncate=False)

## 4. Quick summary theo taxi type

In [ ]:
taxi_demo.groupBy("taxi_type").agg(
    spark_min("pickup_time").alias("min_pickup_time"),
    spark_max("pickup_time").alias("max_pickup_time"),
    spark_round(avg("trip_distance"), 2).alias("avg_trip_distance"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare_amount"),
).show(truncate=False)